In [1]:
!pip -q install tensorflow pandas numpy

In [2]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import files

In [3]:
print("Upload:")
print("1. adult_tf_best_model.tflite")
print("2. adult_preprocessing_assets.json")
print("3. child_tf_best_model.tflite")
print("4. child_preprocessing_assets.json")
print("5. maternal_tf_best_model.tflite")
print("6. maternal_preprocessing_assets.json")

uploaded = files.upload()

uploaded_files = list(uploaded.keys())
print("\nUploaded files:")
for f in uploaded_files:
    print("-", f)

Upload:
1. adult_tf_best_model.tflite
2. adult_preprocessing_assets.json
3. child_tf_best_model.tflite
4. child_preprocessing_assets.json
5. maternal_tf_best_model.tflite
6. maternal_preprocessing_assets.json


Saving adult_preprocessing_assets.json to adult_preprocessing_assets.json
Saving adult_tf_best_model.tflite to adult_tf_best_model.tflite
Saving child_preprocessing_assets.json to child_preprocessing_assets.json
Saving child_tf_best_model.tflite to child_tf_best_model.tflite
Saving maternal_preprocessing_assets.json to maternal_preprocessing_assets.json
Saving maternal_tf_best_model.tflite to maternal_tf_best_model.tflite

Uploaded files:
- adult_preprocessing_assets.json
- adult_tf_best_model.tflite
- child_preprocessing_assets.json
- child_tf_best_model.tflite
- maternal_preprocessing_assets.json
- maternal_tf_best_model.tflite


In [4]:
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def load_tflite_interpreter(model_path):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    return interpreter

adult_assets = load_json("adult_preprocessing_assets.json")
child_assets = load_json("child_preprocessing_assets.json")
maternal_assets = load_json("maternal_preprocessing_assets.json")

adult_interpreter = load_tflite_interpreter("adult_tf_best_model.tflite")
child_interpreter = load_tflite_interpreter("child_tf_best_model.tflite")
maternal_interpreter = load_tflite_interpreter("maternal_tf_best_model.tflite")

print("All models and preprocessing assets loaded successfully.")

All models and preprocessing assets loaded successfully.


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [5]:
def standardize_numeric(values, mean_list, scale_list):
    values = np.array(values, dtype=np.float32)
    mean = np.array(mean_list, dtype=np.float32)
    scale = np.array(scale_list, dtype=np.float32)
    return (values - mean) / scale

def one_hot_encode_row(row_dict, categorical_cols, ohe_categories):
    encoded = []

    for col in categorical_cols:
        value = str(row_dict[col]).strip()
        categories = ohe_categories[col]

        vec = [0.0] * len(categories)
        if value in categories:
            vec[categories.index(value)] = 1.0

        encoded.extend(vec)

    return np.array(encoded, dtype=np.float32)

def preprocess_input(sample_dict, assets):
    numeric_cols = assets["numeric_columns"]
    categorical_cols = assets["categorical_columns"]

    scaler_mean = assets["scaler_mean"]
    scaler_scale = assets["scaler_scale"]
    ohe_categories = assets["ohe_categories"]

    numeric_values = [float(sample_dict[col]) for col in numeric_cols]
    numeric_scaled = standardize_numeric(numeric_values, scaler_mean, scaler_scale)

    categorical_encoded = one_hot_encode_row(sample_dict, categorical_cols, ohe_categories)

    final_input = np.concatenate([numeric_scaled, categorical_encoded]).astype(np.float32)
    final_input = final_input.reshape(1, -1)

    return final_input

In [6]:
def run_tflite_inference(interpreter, X_input):
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    interpreter.set_tensor(input_details[0]["index"], X_input.astype(np.float32))
    interpreter.invoke()

    outputs = [interpreter.get_tensor(out["index"]) for out in output_details]

    # Sort outputs by last dimension:
    # severity -> shape (?, 3)
    # disposition -> shape (?, 4)
    outputs_sorted = sorted(outputs, key=lambda x: x.shape[-1])

    severity_probs = outputs_sorted[0]
    disposition_probs = outputs_sorted[1]

    return disposition_probs, severity_probs

def decode_prediction(disposition_probs, severity_probs, assets):
    clinical_order = assets["clinical_order"]
    severity_order = assets["severity_order"]

    disp_idx = int(np.argmax(disposition_probs, axis=1)[0])
    sev_idx = int(np.argmax(severity_probs, axis=1)[0])

    return {
        "clinical_disposition": clinical_order[disp_idx],
        "severity_score": severity_order[sev_idx],
        "disposition_probabilities": {
            clinical_order[i]: float(disposition_probs[0][i])
            for i in range(len(clinical_order))
        },
        "severity_probabilities": {
            severity_order[i]: float(severity_probs[0][i])
            for i in range(len(severity_order))
        }
    }

In [7]:
def get_text_input(prompt, allowed_values=None):
    while True:
        value = input(prompt).strip()
        if allowed_values is None or value in allowed_values:
            return value
        print(f"Invalid value. Allowed values: {allowed_values}")

def get_int_input(prompt, min_val=None, max_val=None):
    while True:
        try:
            value = int(input(prompt).strip())
            if (min_val is not None and value < min_val) or (max_val is not None and value > max_val):
                print(f"Value must be between {min_val} and {max_val}")
                continue
            return value
        except:
            print("Invalid integer input.")

def get_float_input(prompt, min_val=None, max_val=None):
    while True:
        try:
            value = float(input(prompt).strip())
            if (min_val is not None and value < min_val) or (max_val is not None and value > max_val):
                print(f"Value must be between {min_val} and {max_val}")
                continue
            return value
        except:
            print("Invalid numeric input.")

In [8]:
#Custom data
def collect_adult_input():
    print("\n--- ADULT INPUT ---")

    sample = {
        'age_years': get_int_input("Age in years (0-120): ", 0, 120),
        'sex': get_text_input("Sex [Male/Female]: ", ["Male", "Female"]),
        'heart_rate_bpm': get_int_input("Heart rate bpm (30-220): ", 30, 220),
        'respiratory_rate_bpm': get_int_input("Respiratory rate bpm (5-60): ", 5, 60),
        'systolic_bp_mmHg': get_int_input("Systolic BP mmHg (50-250): ", 50, 250),
        'spo2_percent': get_int_input("SpO2 percent (50-100): ", 50, 100),
        'temperature_c': get_float_input("Temperature C (30-45): ", 30, 45),
        'level_of_consciousness': get_text_input(
            "Level of consciousness [Alert/Voice/Pain/Unresponsive]: ",
            ["Alert", "Voice", "Pain", "Unresponsive"]
        ),
        'chief_complaint_category': get_text_input(
            "Chief complaint [Respiratory/Cardiac/Infection/Trauma/Other]: ",
            ["Respiratory", "Cardiac", "Infection", "Trauma", "Other"]
        ),
        'duration_days': get_int_input("Duration in days (0-60): ", 0, 60),
        'comorbidity_count': get_int_input("Comorbidity count (0-10): ", 0, 10),
        'pain_distress_score_0_10': get_int_input("Pain/distress score (0-10): ", 0, 10)
    }

    return sample
def collect_child_input():
    print("\n--- CHILD INPUT ---")

    sample = {
        'age_months': get_int_input("Age in months (0-60): ", 0, 60),
        'weight_kg': get_float_input("Weight in kg (1-40): ", 1, 40),
        'fever_present': get_text_input("Fever present [Yes/No]: ", ["Yes", "No"]),
        'fever_duration_days': get_int_input("Fever duration in days (0-30): ", 0, 30),
        'respiratory_rate_bpm': get_int_input("Respiratory rate bpm (5-100): ", 5, 100),
        'chest_indrawing': get_text_input("Chest indrawing [Yes/No]: ", ["Yes", "No"]),
        'ability_to_drink_feed': get_text_input(
            "Ability to drink/feed [Normal/Reduced/Unable]: ",
            ["Normal", "Reduced", "Unable"]
        ),
        'vomiting_everything': get_text_input("Vomiting everything [Yes/No]: ", ["Yes", "No"]),
        'convulsions': get_text_input("Convulsions [Yes/No]: ", ["Yes", "No"]),
        'lethargic_or_unconscious': get_text_input("Lethargic or unconscious [Yes/No]: ", ["Yes", "No"]),
        'diarrhea_duration_days': get_int_input("Diarrhea duration in days (0-30): ", 0, 30),
        'dehydration_signs': get_text_input(
            "Dehydration signs [None/Some/Severe]: ",
            ["None", "Some", "Severe"]
        ),
        'spo2_percent': get_int_input("SpO2 percent (50-100): ", 50, 100),
        'malnutrition_indicator': get_text_input("Malnutrition indicator [Yes/No]: ", ["Yes", "No"])
    }

    return sample
def collect_maternal_input():
    print("\n--- MATERNAL INPUT ---")

    sample = {
        'age_years': get_int_input("Age in years (10-60): ", 10, 60),
        'gestational_age_weeks': get_int_input("Gestational age in weeks (0-45): ", 0, 45),
        'systolic_bp_mmHg': get_int_input("Systolic BP mmHg (50-250): ", 50, 250),
        'heart_rate_bpm': get_int_input("Heart rate bpm (30-220): ", 30, 220),
        'vaginal_bleeding': get_text_input("Vaginal bleeding [Yes/No]: ", ["Yes", "No"]),
        'severe_headache_or_vision_issues': get_text_input(
            "Severe headache or vision issues [Yes/No]: ",
            ["Yes", "No"]
        ),
        'abdominal_pain_severity_0_10': get_int_input("Abdominal pain severity (0-10): ", 0, 10),
        'fetal_movement': get_text_input(
            "Fetal movement [Normal/Reduced/Absent]: ",
            ["Normal", "Reduced", "Absent"]
        ),
        'fever_present': get_text_input("Fever present [Yes/No]: ", ["Yes", "No"]),
        'seizures': get_text_input("Seizures [Yes/No]: ", ["Yes", "No"]),
        'previous_complications': get_text_input("Previous complications [Yes/No]: ", ["Yes", "No"]),
        'hemoglobin_g_dL': get_float_input("Hemoglobin g/dL (3-20): ", 3, 20),
        'edema': get_text_input("Edema [Yes/No]: ", ["Yes", "No"]),
        'duration_days': get_int_input("Duration in days (0-60): ", 0, 60)
    }

    return sample


In [9]:
def predict_for_patient_type(patient_type):
    if patient_type == "adult":
        assets = adult_assets
        interpreter = adult_interpreter
        sample = collect_adult_input()

    elif patient_type == "child":
        assets = child_assets
        interpreter = child_interpreter
        sample = collect_child_input()

    elif patient_type == "maternal":
        assets = maternal_assets
        interpreter = maternal_interpreter
        sample = collect_maternal_input()

    else:
        raise ValueError("Invalid patient type")

    X_input = preprocess_input(sample, assets)
    disposition_probs, severity_probs = run_tflite_inference(interpreter, X_input)
    result = decode_prediction(disposition_probs, severity_probs, assets)

    print("\n" + "="*60)
    print("PREDICTION RESULT")
    print("="*60)
    print("clinical_disposition :", result["clinical_disposition"])
    print("severity_score       :", result["severity_score"])

    print("\nDisposition probabilities:")
    for k, v in result["disposition_probabilities"].items():
        print(f"{k}: {v:.4f}")

    print("\nSeverity probabilities:")
    for k, v in result["severity_probabilities"].items():
        print(f"{k}: {v:.4f}")

    return result

In [10]:
print("Available patient types:")
print("- adult")
print("- child")
print("- maternal")

patient_type = input("Enter patient type: ").strip().lower()

result = predict_for_patient_type(patient_type)

Available patient types:
- adult
- child
- maternal
Enter patient type: adult

--- ADULT INPUT ---
Age in years (0-120): 23
Sex [Male/Female]: Male
Heart rate bpm (30-220): 45
Respiratory rate bpm (5-60): 67
Value must be between 5 and 60
Respiratory rate bpm (5-60): 49
Systolic BP mmHg (50-250): 77
SpO2 percent (50-100): 56
Temperature C (30-45): 45
Level of consciousness [Alert/Voice/Pain/Unresponsive]: Voice
Chief complaint [Respiratory/Cardiac/Infection/Trauma/Other]: Trauma
Duration in days (0-60): 2
Comorbidity count (0-10): 4
Pain/distress score (0-10): 6

PREDICTION RESULT
clinical_disposition : Emergency referral
severity_score       : High

Disposition probabilities:
Treat locally: 0.0000
Treat + monitor: 0.0000
Stabilize + refer: 0.0000
Emergency referral: 1.0000

Severity probabilities:
Low: 0.0000
Medium: 0.0000
High: 1.0000


In [11]:
print("Available patient types:")
print("- adult")
print("- child")
print("- maternal")

patient_type = input("Enter patient type: ").strip().lower()

result = predict_for_patient_type(patient_type)

Available patient types:
- adult
- child
- maternal
Enter patient type: child

--- CHILD INPUT ---
Age in months (0-60): 34
Weight in kg (1-40): 7
Fever present [Yes/No]: No
Fever duration in days (0-30): 4
Respiratory rate bpm (5-100): 45
Chest indrawing [Yes/No]: Yes
Ability to drink/feed [Normal/Reduced/Unable]: Unable
Vomiting everything [Yes/No]: Yes
Convulsions [Yes/No]: No
Lethargic or unconscious [Yes/No]: No
Diarrhea duration in days (0-30): 3
Dehydration signs [None/Some/Severe]: None
SpO2 percent (50-100): 56
Malnutrition indicator [Yes/No]: No

PREDICTION RESULT
clinical_disposition : Emergency referral
severity_score       : High

Disposition probabilities:
Treat locally: 0.0000
Treat + monitor: 0.0000
Stabilize + refer: 0.0000
Emergency referral: 1.0000

Severity probabilities:
Low: 0.0000
Medium: 0.0000
High: 1.0000


In [12]:
print("Available patient types:")
print("- adult")
print("- child")
print("- maternal")

patient_type = input("Enter patient type: ").strip().lower()

result = predict_for_patient_type(patient_type)

Available patient types:
- adult
- child
- maternal
Enter patient type: maternal

--- MATERNAL INPUT ---
Age in years (10-60): 45
Gestational age in weeks (0-45): 56
Value must be between 0 and 45
Gestational age in weeks (0-45): 45
Systolic BP mmHg (50-250): 78
Heart rate bpm (30-220): 89
Vaginal bleeding [Yes/No]: No
Severe headache or vision issues [Yes/No]: No
Abdominal pain severity (0-10): 8
Fetal movement [Normal/Reduced/Absent]: Absent
Fever present [Yes/No]: Yes
Seizures [Yes/No]: No
Previous complications [Yes/No]: No
Hemoglobin g/dL (3-20): 5
Edema [Yes/No]: No
Duration in days (0-60): 9

PREDICTION RESULT
clinical_disposition : Emergency referral
severity_score       : High

Disposition probabilities:
Treat locally: 0.0000
Treat + monitor: 0.0000
Stabilize + refer: 0.0040
Emergency referral: 0.9960

Severity probabilities:
Low: 0.0000
Medium: 0.0014
High: 0.9986
